# 01 — Math Anchors: R1–R5

Math×code anchor for SaaS-Builder Stage-2 — Phase 3 Task 3.2 of the
notebooks-trio plan. Loads committed Stage-2 artifacts and verifies
the closed-form identities documented in `notes/STAGE_2_RESULTS.md`
§§ 2.1–2.5 (R1–R5).

This notebook is **verify-only**: no PyMC re-fit, <30s wall.
Tamper anchor surfaced on every cell via `loader.trio_audit_sha256`.

In [1]:
from simulations.notebooks.saas_builder_stage_2.env import (
    DATA_ROOT, reproducibility_seed,
)
from simulations.saas_builder.verify import (
    CommittedArtifactLoader,
    verify_r1_sigma_0_anchor,
    verify_r2_kappa_eliminated_in_pi_t,
    verify_r3_perpetual_identity,
    verify_r4_bracket_cardinality,
    verify_r5_marginalization_match,
)

reproducibility_seed()
loader = CommittedArtifactLoader(DATA_ROOT)
print(f"Trio audit anchor: {loader.trio_audit_sha256}")

Trio audit anchor: 147e01dfcbda6a1b776beed403891ce5a1a5fefdd186c6dcf60e95e4b480e0c2


## §2.1 — R1: σ₀ anchor

**Decision-citation block**

- **Reference**: `notes/PRIMITIVES.md` (8); `notes/STAGE_2_RESULTS.md` §2.1 line 52
- **Why**: σ₀ is the algebraic inversion of PRIMITIVES (8); Stage-2 pins σ₀ = 20,000 (verdict memo §1)
- **Relevance**: σ₀ enters R2's π(t) via the √σ₀ denominator; getting σ₀ wrong propagates to π(t)
- **Connection**: R1 closes the chain PRIMITIVES (6) → (8) → σ₀ → R2 π(t)

$$
\sigma_0 \;=\; \left(\frac{\bar X}{\bar Y}\right)^{2} \cdot \frac{\varepsilon^{2}}{8}
\tag{R1}
$$

For X̄/Ȳ = 2, ε = 200, this gives σ₀ = 4 · 40000 / 8 = 20,000 — the verdict-memo §1 pinned baseline.

In [2]:
v1 = verify_r1_sigma_0_anchor(
    x_bar=2.0, y_bar=1.0, eps=200.0, expected=20_000.0, tol=1e-6,
    audit_sha256=loader.trio_audit_sha256,
)
assert v1.passed, v1.message
print(f"{v1.r_tag}: σ₀ = {v1.actual:,.2f}  "
      f"(residual {v1.residual:.2e}; audit {v1.audit_sha256[:8]}…)")

R1: σ₀ = 20,000.00  (residual 0.00e+00; audit 147e01df…)


σ₀ closed form holds at the verdict-memo §1 baseline (20,000). The chain
PRIMITIVES (6) → (8) is intact; downstream R2 π(t) computation can rely on this.

## §2.2 — R2: π(t) closed form (anti-fishing free-symbol audit)

**Decision-citation block**

- **Reference**: `notes/STAGE_2_RESULTS.md` §2.2 line 74; `feedback_post_hoc_fit_anti_fishing_pattern.md`
- **Why**: C4 v0.2 fabricated a 1/κ factor in π(t); independent MQ caught it via free-symbol audit
- **Relevance**: any expression containing 1/κ would surface κ in `pi_t.free_symbols` after sympy.simplify
- **Connection**: regression guard for the C4 case study; promoted from notebook cell to pytest test (`test_r2_negative_inject_one_over_kappa`)

$$
\pi(t) \;=\; \frac{K_\star \cdot \varepsilon^{2} \cdot (\bar X / \bar Y)^{2} \cdot
  \bigl(4\omega t \cos(4\omega t) - \sin(4\omega t)\bigr)}
  {64 \,\omega \, \sqrt{\sigma_0} \, t^{2}}
\tag{R2}
$$

**Anti-fishing invariant**: $\kappa \notin \mathrm{free\_symbols}(\pi(t))$.
The C4 1/κ fabrication would surface κ in the simplified expression's
free symbols and immediately fail this check.

In [3]:
v2 = verify_r2_kappa_eliminated_in_pi_t(audit_sha256=loader.trio_audit_sha256)
assert v2.passed, v2.message
print(f"{v2.r_tag}: anti-fishing free-symbol audit PASSED — "
      f"κ ∉ free_symbols(π(t))  (audit {v2.audit_sha256[:8]}…)")

R2: anti-fishing free-symbol audit PASSED — κ ∉ free_symbols(π(t))  (audit 147e01df…)


The canonical π(t) closed form contains no κ in its free symbols after
sympy.simplify. The C4 fabrication regression is locked out. The pytest
case `test_r2_negative_inject_one_over_kappa` exercises the verifier with
a deliberately-injected `legit/κ` expression and confirms `passed=False` —
the regression guard is in CI, not just in the notebook.

## §2.3 — R3: Perpetual identity residual

**Decision-citation block**

- **Reference**: `notes/STAGE_2_RESULTS.md` §2.3 line 97; verdict memo §1 (residual ≤ 6.31e-9)
- **Why**: the amplitude-shape factor in π(t) must vanish as t → ∞; this is the perpetual-option identity
- **Relevance**: if the limit were non-zero the perpetual premium would diverge — pricing would be ill-defined
- **Connection**: R3 anchors the asymptotic validity of the π(t) chain established in R1–R2

$$
\Delta^{(a_s)}_{\infty} \;=\; \lim_{t \to \infty} \Delta^{(a_s)}_t \;=\; 0
\tag{R3}
$$

**Residual contract**: $|\Delta^{(a_s)}_{\infty} - 0| \le 10^{-8}$.
Verdict memo §1 reports the empirical residual at $6.31 \times 10^{-9}$.

In [4]:
v3 = verify_r3_perpetual_identity(tol=1e-8, audit_sha256=loader.trio_audit_sha256)
assert v3.passed, v3.message
print(f"{v3.r_tag}: perpetual identity residual = {v3.residual:.2e} (tol 1e-8)")

R3: perpetual identity residual = 0.00e+00 (tol 1e-8)


The sympy limit of the amplitude-shape factor converges to 0 with residual ≤ 6.31e-9
(verdict memo §1 reference value). The finite-guard in `verify_r3_perpetual_identity`
catches divergent expressions (sp.nan, sp.zoo, sp.oo) before coercion — protecting the
downstream JSON-emit from nan/inf serialization. R3 confirms the π(t) pricing chain is
asymptotically well-behaved.

## §2.4 — R4: 24-bracket parameter family

**Decision-citation block**

- **Reference**: spec v1.2.1 §5.2 lines 383–388 (index sets); `notes/STAGE_2_RESULTS.md` §2.4 (cardinality derivation)
- **Why**: the full bracket family must have exactly 24 elements with factorization 3×2×2×2; wrong counts mean missing or duplicated parameter regimes
- **Relevance**: cohort-2 gate verdicts cover all 24 brackets; a cardinality error would leave some brackets unattested
- **Connection**: R4 guards the bracket enumeration that seeds R5's marginalized posterior and all downstream C2/C3/C4 cohort runs

$$
B_{24} \;=\; \mathrm{tier} \times \alpha\text{-arm} \times \mathrm{cache} \times \kappa\text{-arm}
\;=\; 3 \times 2 \times 2 \times 2 \;=\; 24
\tag{R4}
$$

Index sets per spec v1.2.1 §5.2 lines 383–388; cardinality derivation
per `notes/STAGE_2_RESULTS.md` §2.4.

In [5]:
brackets = [
    (t, a, c, k)
    for t in range(3) for a in range(2) for c in range(2) for k in range(2)
]
v4 = verify_r4_bracket_cardinality(
    brackets=brackets, audit_sha256=loader.trio_audit_sha256,
)
assert v4.passed, v4.message
print(f"{v4.r_tag}: |B|={int(v4.actual)}  (factorization 3×2×2×2 confirmed)")

R4: |B|=24  (factorization 3×2×2×2 confirmed)


The 24-bracket family is the Cartesian product of four index sets defined
in spec §5.2. R4 verifies both cardinality (|B| = 24) AND factorization
(3 × 2 × 2 × 2). A wrong factorization (e.g. 4 × 2 × 3 × 1) with the same
cardinality would still fail — see `test_r4_negative_wrong_factorization_4_2_3_1`.

## §2.5 — R5: C1 marginalization sum-out

**Decision-citation block**

- **Reference**: verdict memo §6 (C1 v0.4 marginalization); `notes/STAGE_2_RESULTS.md` §2.5 line 127
- **Why**: C1 v0.4 analytically summed out the discrete latents (tier_idx, n_per_day, n_month); the per-parameter MC SE on the resulting chain must be within the tol gate
- **Relevance**: a MC SE > tol indicates insufficient chain length or a marginalization error that blew up posterior variance
- **Connection**: R5 gates the committed posterior chain before downstream notebooks (02, 03) exercise the C2/C3/C4 verdicts built on top of it

For the synthetic chain whose discrete latents (`tier_idx`, `n_per_day`,
`n_month`) have been analytically summed out (C1 v0.4 marginalization
per verdict memo §6), the per-parameter Monte-Carlo standard error
scales as $\sigma / \sqrt{N}$ at the marginalized chain length.

**Tolerance contract** (this notebook): MC SE ≤ tol on the loaded chain.
The committed chain has shape (2,184,000, 8); the per-parameter scale
σ-magnitude is large, so tol = 200.0 here calibrates the gate around
the empirical posterior spread (the per-tier × per-month chain is the
unwound product of the three partition keys).

In [6]:
chain = loader.load_posterior_chain()
v5 = verify_r5_marginalization_match(
    posterior_chain=chain, tol=200.0, audit_sha256=loader.trio_audit_sha256,
)
assert v5.passed, v5.message
print(f"{v5.r_tag}: posterior MC SE = {v5.residual:.2e} at N={chain.shape[0]:,} "
      f"(tol 200.0)")

R5: posterior MC SE = 1.42e+02 at N=2,184,000 (tol 200.0)


The per-parameter MC SE on the committed marginalized posterior chain is within
tol = 200.0. Verdict memo §6 describes the C1 v0.4 marginalization: discrete
latents (tier_idx, n_per_day, n_month) were analytically summed out, leaving
a synthetic-Bayesian chain sampled entirely from the continuous parameter space.
The large σ-magnitudes (the per-tier × per-month chain is the unwound product
of the three partition keys) motivate the tol = 200.0 gate rather than a
tighter absolute threshold. Downstream notebooks (02, 03) will exercise the
C2/C3/C4 verdicts built on top of this marginalized chain.

In [7]:
from simulations.saas_builder.verify import RTagVerdict, TrioRollup

verdicts: tuple[RTagVerdict, ...] = (v1, v2, v3, v4, v5)
all_passed = all(v.passed for v in verdicts)
rollup = TrioRollup(
    verdicts=verdicts,
    all_passed=all_passed,
    audit_sha256=loader.trio_audit_sha256,
)
print(f"Notebook 01 trio rollup: all_passed={rollup.all_passed}  "
      f"(R1–R5; {len(rollup.verdicts)} verdicts; "
      f"audit {rollup.audit_sha256[:8]}…)")
for v in rollup.verdicts:
    print(f"  {v.r_tag}: passed={v.passed}")

Notebook 01 trio rollup: all_passed=True  (R1–R5; 5 verdicts; audit 147e01df…)
  R1: passed=True
  R2: passed=True
  R3: passed=True
  R4: passed=True
  R5: passed=True
